[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/next-gen-aiam/blob/main/notebooks/05b_cross_asset_deep_momentum.ipynb)

# Notebook 05b — Cross-Asset Deep Momentum (the §5.4 extension)

*Direct-weight (BSV-2009) deep momentum on the 29-asset 2003–2026 universe, using delta-of-straddle momentum features, a **negative-Sharpe** portfolio-level loss,
pooled-asset-wise vs cross-asset MLP architectures, monthly walk-forward retrain on a rolling
10-year window, and a 10-seed ensemble.*

## Abstract

This notebook closes a loose end from **Notebook 05**. Notebook 05 implemented the
Brandt–Santa-Clara–Valkanov (2009) *direct-weight* policy — the network outputs portfolio
weights and is trained end-to-end on a portfolio-level loss — but deliberately on the **least
favorable** universe for deep learning: 2 assets (SPY + IEF), where there is almost no
cross-sectional structure to learn. The result there was a null (best DL Sharpe ≈ 1.240 vs
Risk Parity 1.247). Its §5.4 named the obvious next step: *run the same machinery on the full
29-asset cross-asset universe.* The §5.4 extension runs that same direct-weight / Sharpe-loss engine on the **full 29-asset
cross-asset universe**, combining notebook 04's direct-Sharpe objective with the cross-asset MLP
architecture and delta-of-straddle inputs — all scored on our apples-to-apples harness.

**The question this answers:** was the notebook-05 null about the *method* (direct-weight DL
doesn't help) or about the *2-asset cage* (no cross-sectional signal to exploit)?

## Where this sits in the cross-paradigm story

| Notebook | Paradigm | Universe | Result |
|---|---|---|---|
| 03 | ML ensemble (Lasso/RF/XGB → MSR) | 29-asset | **Wins** — Sharpe 2.579 (2023–26 OOS) |
| 04 | DL predict-then-optimize (DL → μ̂ → MSR) | 29-asset | Misses — best ≈ 2.386 |
| 05 | DL direct-weight (BSV 2009) | **2-asset** | Ties Risk Parity (−0.008) |
| **05b** | **DL direct-weight, cross-asset deep momentum** | **29-asset** | *this notebook* |

**Pre-registration / discipline note.** To keep this comparable to the rest of the repo and to
avoid manufacturing an overfit, the architecture and training recipe are **fixed before any OOS result is seen** (see §3–§5). No per-seed or per-architecture search to
chase a number. Single-pass, reported against the same harness and benchmarks as notebooks 00–05.

**Honest priors (from the prior chat / gut-feel calibration):**
- Beats the nb-05 null / ties-or-beats Risk Parity and TSMOM *in this universe and window*: likely.
- Cross-asset variant beats the pooled variant (the cross-asset-signal hypothesis): roughly even-to-likely.
- Clears the 2023–26 classical OOS bar (~2.42): unlikely — that bar is set by covariance/diversification
  strategies, and this is momentum-only.
- Becomes leaderboard #1 (beats 2.579): improbable.

A null *on the favorable cross-asset case* is itself a strong result — a much sharper version of
"complexity does not pay" than the 2-asset null was.

## §T1 — Theory bridge: BSV direct-weight and cross-asset deep momentum

**BSV (2009).** Map characteristics directly to weights, skipping return forecasting:

$$w_{i,t} = f_\theta(x_{i,t}), \qquad \max_\theta\; \tfrac{1}{T}\sum_t U\!\big(R_{p,t+1}\big),\quad
R_{p,t+1}=\sum_i w_{i,t}(\theta)\, r_{i,t+1}.$$

**Cross-asset deep momentum (direct-weight Sharpe loss).** Identical paradigm with $U = \text{Sharpe}$ and an explicit transaction-cost
term. Let $X^{(i)}_t\in[-1,1]$ be the network's position for asset $i$, $\sigma^{(i)}_t$ its
volatility, $c$ the per-unit cost, and $r^{(i)}_{t,t+1}$ the one-month forward return:

$$R_{t,t+1} = \frac{S}{N}\sum_{i=1}^{N}\left(\frac{X^{(i)}_t}{\sigma^{(i)}_t}\,r^{(i)}_{t,t+1}
- c\left|\frac{X^{(i)}_t}{\sigma^{(i)}_t}\right|\right),
\qquad \text{Loss} = -\,\frac{\mathbb{E}[R]}{\sigma(R)}.$$

Because the loss is the Sharpe of the *realized strategy return over the holding month*, the
network is optimized for risk-adjusted performance directly and the holding-period ambiguity of
classical momentum is resolved implicitly. $S$ scales to 10% annualized vol on an expanding window
— it is irrelevant to the (scale-invariant) Sharpe loss and applied only at backtest time.

**Delta-of-straddle inputs.** The approach feeds the *un-averaged* delta-of-straddle signal at five
lookbacks (32/64/126/252/504d) so the network learns the horizon weighting itself, rather than
a legacy fixed average. The signal is equivalent to testing whether the mean return
is significantly positive or negative, and lives in $[-1,1]$. We use that t-stat proxy:

$$t^{(i)}_{L} = \sqrt{L}\;\frac{\overline{r}^{(i)}_{L}}{s^{(i)}_{L}},\qquad
\text{signal}^{(i)}_{L} = 2\,\Phi\!\big(t^{(i)}_{L}\big) - 1 \in [-1,1],$$

where $\Phi$ is the standard-normal CDF (so larger trends saturate, encoding "outsized returns are
less probable" — the same shape the option-delta gives).

## §0 — Setup (Colab-aware)

Clones the repo *for data only* if we're not already inside it. The model/training/walk-forward
engine is defined in this notebook (no dependency on `src/aiam/` DL internals), mirroring the
notebook-05 wrapper precedent.

> If `next-gen-aiam` is private, either run this notebook from inside a local checkout, or
> authenticate the clone (e.g. a tokenized URL) in the cell below.

In [ ]:
import os, sys, subprocess
from pathlib import Path

def find_root(start: Path) -> Path | None:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "data").exists():
            return p
    return None

ROOT = find_root(Path.cwd().resolve())
IN_COLAB = "google.colab" in sys.modules

if ROOT is None:
    REPO_URL = os.environ.get("AIAM_REPO_URL", "https://github.com/FranQuant/next-gen-aiam.git")
    dest = Path("/content/next-gen-aiam") if IN_COLAB else Path.cwd() / "next-gen-aiam"
    if not dest.exists():
        print(f"Cloning {REPO_URL} -> {dest} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    ROOT = dest

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"ROOT: {ROOT}")
print(f"In Colab: {IN_COLAB}")


In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

import torch
import torch.nn as nn

TRADING_DAYS = 252
MONTHS_PER_YEAR = 12

# Reuse repo helpers if importable; otherwise define local fallbacks.
try:
    from aiam.data.universe import UNIVERSE_29
except Exception:
    UNIVERSE_29 = None  # resolved from the returns columns below

def ann_sharpe(r):
    r = pd.Series(r).dropna()
    return r.mean() / r.std() * np.sqrt(TRADING_DAYS) if r.std() > 0 else np.nan

def ann_return(r):
    r = pd.Series(r).dropna()
    return (1 + r).prod() ** (TRADING_DAYS / len(r)) - 1 if len(r) else np.nan

def ann_vol(r):
    return pd.Series(r).dropna().std() * np.sqrt(TRADING_DAYS)

def max_drawdown(r):
    cum = (1 + pd.Series(r).dropna()).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}")

# A tiny [20,10,20] MLP on 29 assets does NOT need a big GPU — a free T4 (or even CPU) is fine.


In [ ]:
# ── Run mode & pre-registered config ────────────────────────────────────────
RUN_MODE = "full"          # "smoke" (a few refits, fast sanity) or "full"

# Pre-registered defaults (fixed before seeing OOS results)
CFG = dict(
    lookbacks      = [32, 64, 126, 252, 504],   # delta-of-straddle horizons (days)
    train_years    = 10,        # rolling training window
    refit_stride   = 1,         # 1 = retrain every month; bump to 3 for ~3x faster
    val_frac       = 0.15,      # last 15% of train window -> early-stopping validation
    n_seeds        = 10,        # ensemble size
    max_epochs     = 300,
    patience       = 10,        # early stopping
    batch_size     = 64,
    l2             = 1e-3,      # weight decay
    cost_bps       = 5.0,       # per-unit gross-notional cost in the training loss (bps)
    bt_cost_bps    = 10.0,      # turnover cost applied in the reported backtest (bps)
    target_vol     = 0.10,      # expanding-window vol target for the backtest
    exec_lag_days  = 1,         # 1-day execution lag
    vol_lookback   = 60,        # per-asset vol estimate horizon (days)
    # architecture-specific learning rates
    lr_pooled      = 1e-3,
    lr_cross       = 1e-2,
    # hidden sizes
    pooled_hidden  = [16, 8],   # per-asset MLP (2 hidden layers, 5 inputs -> 1)
    cross_hidden   = [20, 10, 20],  # cross-asset autoencoder-shaped bottleneck
)
if RUN_MODE == "smoke":
    CFG.update(n_seeds=2, max_epochs=40, refit_stride=6)

OUT_DIR = ROOT / "results" / "notebook_05b"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_MODE:", RUN_MODE)
print("OUT_DIR:", OUT_DIR)


## §1 — Data loading

Canonical 29-asset daily price/return cache, plus the published strategy-return tables so we can
score notebook 06 against the existing leaderboard.

In [ ]:
returns = pd.read_parquet(ROOT / "data/cache/returns_29_2003_2026.parquet")
returns.index = pd.to_datetime(returns.index); returns.index.name = "Date"
returns.columns.name = "Asset"

prices = pd.read_parquet(ROOT / "data/cache/prices_29.parquet")
prices.index = pd.to_datetime(prices.index); prices.index.name = "Date"
prices.columns.name = "Asset"

if UNIVERSE_29 is None:
    UNIVERSE_29 = list(returns.columns)
ASSETS = [a for a in UNIVERSE_29 if a in returns.columns]
returns = returns[ASSETS].copy()
prices = prices.reindex(columns=ASSETS)
N = len(ASSETS)
print(f"Returns: {returns.shape}  {returns.index[0].date()} -> {returns.index[-1].date()}")
print(f"Assets ({N}): {ASSETS}")

# Optional: leaderboard / benchmark strategy returns (best-effort; skipped if absent)
def _try_parquet(rel):
    p = ROOT / rel
    if p.exists():
        df = pd.read_parquet(p); df.index = pd.to_datetime(df.index); return df
    print(f"  (missing, skipped) {rel}")
    return None

base_pub = _try_parquet("data/cache/portfolio_returns/31strategies_29assets_2003_2026.parquet")
vmp_pub  = _try_parquet("data/cache/portfolio_returns/31strategies_vmp_29assets_2003_2026.parquet")
ml_ext   = _try_parquet("data/cache/portfolio_returns/ml_strategies_29assets_extended.parquet")


## §2 — Delta-of-straddle momentum features (t-stat proxy)

For each asset and each lookback $L\in\{32,64,126,252,504\}$ we compute, using **only past data**,
the standardized mean-return t-stat and squash it to $[-1,1]$ via $2\Phi(t)-1$. Per-asset volatility
(annualized, `vol_lookback`-day) is stored for the $X/\sigma$ position scaling.

In [ ]:
def delta_straddle_signals(returns: pd.DataFrame, lookbacks) -> dict:
    """Return {L: DataFrame[Date,Asset] of signal in [-1,1]} using only trailing data."""
    sig = {}
    for L in lookbacks:
        mu = returns.rolling(L).mean()
        sd = returns.rolling(L).std(ddof=1)
        tstat = np.sqrt(L) * mu / sd.replace(0.0, np.nan)
        sig[L] = pd.DataFrame(2.0 * norm.cdf(tstat.values) - 1.0,
                              index=returns.index, columns=returns.columns)
    return sig

SIG = delta_straddle_signals(returns, CFG["lookbacks"])
ann_vol_daily = returns.rolling(CFG["vol_lookback"]).std(ddof=1) * np.sqrt(TRADING_DAYS)
ann_vol_daily = ann_vol_daily.clip(lower=1e-4)

# Month-end rebalance dates (last trading day of each month) with full feature coverage.
month_ends = returns.resample("ME").last().index
month_ends = [d for d in returns.index if d in set(returns.resample("ME").apply(lambda x: x.index[-1]))]
month_ends = pd.DatetimeIndex(sorted(returns.groupby(returns.index.to_period("M")).apply(lambda x: x.index[-1]).values))
month_ends = month_ends[month_ends >= returns.index[max(CFG["lookbacks"])]]
print(f"Rebalance month-ends: {len(month_ends)}  {month_ends[0].date()} -> {month_ends[-1].date()}")


In [ ]:
def build_monthly_tensors(month_ends):
    """Assemble aligned monthly arrays.

    X     : (T, N, F)  delta-of-straddle features at each month-end (F = #lookbacks)
    invvol: (T, N)     1/sigma at each month-end (for X/sigma scaling)
    fwd   : (T, N)     next-month forward return (exec lag applied), NaN for last month
    dates : list of month-end Timestamps
    """
    F = len(CFG["lookbacks"])
    dates = list(month_ends)
    T = len(dates)
    X = np.full((T, N, F), np.nan, np.float32)
    invvol = np.full((T, N), np.nan, np.float32)
    fwd = np.full((T, N), np.nan, np.float32)

    idx = returns.index
    lag = CFG["exec_lag_days"]
    for ti, d in enumerate(dates):
        for fi, L in enumerate(CFG["lookbacks"]):
            X[ti, :, fi] = SIG[L].loc[d, ASSETS].values
        invvol[ti] = 1.0 / ann_vol_daily.loc[d, ASSETS].values
        if ti + 1 < T:
            d_next = dates[ti + 1]
            pos = idx.get_indexer([d])[0]
            entry = idx[min(pos + lag, len(idx) - 1)]      # enter `lag` days after signal date
            # forward return = compounded daily returns from entry through next month-end
            seg = returns.loc[entry:d_next, ASSETS]
            fwd[ti] = (1.0 + seg).prod().values - 1.0
    return np.nan_to_num(X, nan=0.0), np.nan_to_num(invvol, nan=0.0), fwd, dates

X_all, INVVOL_all, FWD_all, DATES = build_monthly_tensors(month_ends)
print("X:", X_all.shape, "| invvol:", INVVOL_all.shape, "| fwd:", FWD_all.shape)


## §3 — Architectures

**Pooled asset-wise:** one small MLP *per asset* (5 → 16 → 8 → 1, tanh out), positions pooled into
the portfolio; trained jointly under the pooled-portfolio Sharpe loss. **Cross-asset:** a single MLP
ingesting all assets' features at once (N·5 → 20 → 10 → 20 → N, tanh out) — the autoencoder-shaped
bottleneck lets it learn inter-asset structure. Both use batch-norm → ReLU on hidden layers.

In [ ]:
def mlp_block(sizes, out_dim):
    """Linear -> BN -> ReLU stack, then a final Linear+Tanh to out_dim."""
    layers = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), nn.BatchNorm1d(b), nn.ReLU()]
    layers += [nn.Linear(sizes[-1], out_dim), nn.Tanh()]
    return nn.Sequential(*layers)

class CrossAssetMLP(nn.Module):
    """All momentum features in -> position vector out. Cross-asset model."""
    def __init__(self, n_assets, n_feat, hidden):
        super().__init__()
        self.net = mlp_block([n_assets * n_feat, *hidden], n_assets)
    def forward(self, x):                      # x: (B, N, F)
        b = x.shape[0]
        return self.net(x.reshape(b, -1))      # (B, N) in [-1,1]

class PooledAssetMLP(nn.Module):
    """Independent per-asset MLPs (no weight sharing), positions pooled. Pooled-asset-wise model."""
    def __init__(self, n_assets, n_feat, hidden):
        super().__init__()
        self.nets = nn.ModuleList([mlp_block([n_feat, *hidden], 1) for _ in range(n_assets)])
    def forward(self, x):                      # x: (B, N, F)
        outs = [self.nets[i](x[:, i, :]) for i in range(len(self.nets))]
        return torch.cat(outs, dim=1)          # (B, N)

def make_model(kind):
    if kind == "cross":
        return CrossAssetMLP(N, len(CFG["lookbacks"]), CFG["cross_hidden"])
    return PooledAssetMLP(N, len(CFG["lookbacks"]), CFG["pooled_hidden"])


## §4 — Negative-Sharpe loss with transaction cost

Implements the per-month strategy return $R_{t}=\frac1N\sum_i\big(\frac{X_i}{\sigma_i}r_i - c|\frac{X_i}{\sigma_i}|\big)$
(the $S$ scaler is dropped — Sharpe is scale-invariant) and returns $-\mathbb{E}[R]/\sigma(R)$,
annualized for readability.

In [ ]:
def sharpe_loss(positions, invvol, fwd, cost):
    """positions,(invvol,fwd): (B,N) tensors. cost: per-unit gross-notional (decimal)."""
    w = positions * invvol                               # X/sigma
    gross = w.abs().sum(dim=1, keepdim=True) + 1e-8
    w = w / gross                                         # normalize gross notional to 1
    pnl = (w * fwd).sum(dim=1) - cost * w.abs().sum(dim=1)
    mu, sd = pnl.mean(), pnl.std(unbiased=False) + 1e-8
    sharpe = mu / sd * np.sqrt(MONTHS_PER_YEAR)
    return -sharpe


## §5 — Walk-forward training (rolling 10y, monthly refit, 10-seed ensemble)

At each rebalance month $d$: train on the trailing `train_years` of monthly samples (last `val_frac`
held out for early stopping), average the positions of `n_seeds` independently trained models, and
record the realized portfolio return for the month $d\!\to\!d{+}1$. Monthly refit;
`refit_stride>1` reuses the last fit to save time.

In [ ]:
def train_one(kind, Xtr, IVtr, FWtr, Xva, IVva, FWva, lr, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(kind).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=CFG["l2"])
    cost = CFG["cost_bps"] / 1e4
    Xtr_t = torch.tensor(Xtr, device=DEVICE); IVtr_t = torch.tensor(IVtr, device=DEVICE)
    FWtr_t = torch.tensor(FWtr, device=DEVICE)
    Xva_t = torch.tensor(Xva, device=DEVICE); IVva_t = torch.tensor(IVva, device=DEVICE)
    FWva_t = torch.tensor(FWva, device=DEVICE)
    n = Xtr.shape[0]; bs = min(CFG["batch_size"], n)
    best_val, best_state, bad = np.inf, None, 0
    for ep in range(CFG["max_epochs"]):
        model.train(); perm = torch.randperm(n, device=DEVICE)
        for s in range(0, n, bs):
            j = perm[s:s + bs]
            if len(j) < 2:  # Sharpe needs >=2 samples for a std
                continue
            opt.zero_grad()
            loss = sharpe_loss(model(Xtr_t[j]), IVtr_t[j], FWtr_t[j], cost)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vloss = sharpe_loss(model(Xva_t), IVva_t, FWva_t, cost).item()
        if vloss < best_val - 1e-5:
            best_val, best_state, bad = vloss, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= CFG["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict_positions(model, Xd):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(Xd[None], device=DEVICE)        # (1, N, F)
        return model(x).cpu().numpy()[0]                 # (N,)


In [ ]:
def run_walkforward(kind):
    lr = CFG["lr_cross"] if kind == "cross" else CFG["lr_pooled"]
    T = len(DATES)
    train_min = CFG["train_years"] * MONTHS_PER_YEAR
    # first rebalance index with a full training window behind it
    start = train_min + 1
    if RUN_MODE == "smoke":
        start = max(start, T - 8)                         # only the last few months
    refit_dates_idx = list(range(start, T - 1))
    print(f"[{kind}] rebalances: {len(refit_dates_idx)} "
          f"({DATES[refit_dates_idx[0]].date()} -> {DATES[refit_dates_idx[-1]].date()})  "
          f"x {CFG['n_seeds']} seeds")

    rows, cached_models, last_fit = [], None, -10**9
    for k, ti in enumerate(refit_dates_idx):
        lo = ti - train_min
        idx_tr = np.arange(lo, ti)                        # strictly before rebalance month
        n_val = max(2, int(len(idx_tr) * CFG["val_frac"]))
        tr, va = idx_tr[:-n_val], idx_tr[-n_val:]

        if (ti - last_fit) >= CFG["refit_stride"] or cached_models is None:
            cached_models = [
                train_one(kind, X_all[tr], INVVOL_all[tr], FWD_all[tr],
                          X_all[va], INVVOL_all[va], FWD_all[va], lr, seed)
                for seed in range(CFG["n_seeds"])
            ]
            last_fit = ti

        pos = np.mean([predict_positions(m, X_all[ti]) for m in cached_models], axis=0)  # (N,)
        w = pos * INVVOL_all[ti]
        w = w / (np.abs(w).sum() + 1e-8)                  # gross-normalized weights
        rows.append((DATES[ti], w))
        if (k + 1) % 25 == 0:
            print(f"  {k+1}/{len(refit_dates_idx)} done")
    wdf = pd.DataFrame([r[1] for r in rows], index=[r[0] for r in rows], columns=ASSETS)
    return wdf

weights = {}
for kind in (["cross"] if RUN_MODE == "smoke" else ["pooled", "cross"]):
    weights[kind] = run_walkforward(kind)
print("Done. Weight frames:", {k: v.shape for k, v in weights.items()})


## §6 — Backtest (daily P&L, turnover cost, expanding 10% vol target)

Hold each month's gross-normalized weights over the following month, charge a one-way turnover cost
at `bt_cost_bps`, then rescale by an expanding-window scalar $S$ so realized vol targets 10%.

In [ ]:
def backtest(wdf, cost_bps, target_vol):
    reb = wdf.index
    daily_w = wdf.reindex(returns.index, method="ffill").reindex(columns=ASSETS)
    daily_w = daily_w.loc[reb[0]:].fillna(0.0)
    r = returns.reindex(daily_w.index)[ASSETS]
    gross = (daily_w.shift(1) * r).sum(axis=1)            # 1-day held weights
    turnover = daily_w.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (cost_bps / 1e4)
    raw = (gross - cost).dropna()
    # expanding-window vol target (no look-ahead: scalar uses only past realized vol)
    exp_vol = raw.expanding(min_periods=CFG["vol_lookback"]).std().shift(1) * np.sqrt(TRADING_DAYS)
    S = (target_vol / exp_vol).clip(0.1, 5.0).fillna(1.0)
    return (raw * S).rename("ret")

strat_ret = {k: backtest(v, CFG["bt_cost_bps"], CFG["target_vol"]) for k, v in weights.items()}
labels = {"pooled": "DeepMom(Pooled)", "cross": "DeepMom(CrossAsset)"}
for k, r in strat_ret.items():
    print(f"{labels[k]:24s} Sharpe={ann_sharpe(r):.3f}  AnnRet={ann_return(r):.1%}  "
          f"Vol={ann_vol(r):.1%}  MaxDD={max_drawdown(r):.1%}  ({r.index[0].date()}->{r.index[-1].date()})")


## §7 — Benchmarks & leaderboard comparison

Two reference points: (1) **in-universe, same-window** benchmarks computed on the strategy's own OOS
span — Risk Parity (the nb-05 benchmark) and a 12-month TSMOM trend-follower (the standard trend-following analogue); (2) the existing **leaderboard bars** for context, with an explicit
window caveat.

In [ ]:
# In-universe benchmarks computed over the deep-momentum OOS span
span = strat_ret["cross"].index
roll_vol = returns[ASSETS].rolling(21).std().shift(1)
inv_vol = (1.0 / roll_vol.clip(lower=1e-8))
rp_w = inv_vol.div(inv_vol.sum(axis=1), axis=0)
ret_rp = (returns[ASSETS] * rp_w).sum(axis=1).reindex(span).rename("RiskParity-21d")

# 12m TSMOM, vol-scaled, monthly rebalanced (standard trend-following analogue)
mom12 = (1 + returns[ASSETS]).rolling(252).apply(np.prod, raw=True) - 1
tsmom_sig = np.sign(mom12)
tsmom_w = (tsmom_sig * inv_vol)
tsmom_w = tsmom_w.div(tsmom_w.abs().sum(axis=1).clip(lower=1e-8), axis=0)
tsmom_w_m = tsmom_w.reindex(month_ends).reindex(returns.index, method="ffill")
ret_tsmom = (tsmom_w_m.shift(1) * returns[ASSETS]).sum(axis=1).reindex(span).rename("TSMOM-12m")

BENCH = {"RiskParity-21d": ret_rp, "TSMOM-12m": ret_tsmom}
print(f"In-universe benchmarks over {span[0].date()} -> {span[-1].date()}:")
for nm, r in BENCH.items():
    print(f"  {nm:18s} Sharpe={ann_sharpe(r):.3f}  AnnRet={ann_return(r):.1%}  MaxDD={max_drawdown(r):.1%}")

# Leaderboard bars (different windows — context only)
print("\nLeaderboard context (NOTE: windows differ — not strictly comparable):")
print("  MSR(Ensemble_mu)  2.579   (2023-26 OOS, nb03)")
print("  VMP(MDP(LW))      2.422   (2023-26 OOS) / 1.372 (2003-26 full)")


In [ ]:
# Same-window sub-comparison: restrict everything to 2023-01 -> end, the nb03/04 OOS window
sub = pd.Timestamp("2023-01-01")
print(f"2023–{span[-1].year} sub-window (leaderboard-comparable span):")
for k, r in strat_ret.items():
    rr = r.loc[sub:]
    print(f"  {labels[k]:24s} Sharpe={ann_sharpe(rr):.3f}")
for nm, r in BENCH.items():
    print(f"  {nm:24s} Sharpe={ann_sharpe(r.loc[sub:]):.3f}")
if ml_ext is not None:
    print("  (ml_strategies_29assets_extended present — align columns to compare directly if desired)")


## §8 — Plots

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for k, r in strat_ret.items():
    ax.plot((1 + r).cumprod(), lw=1.6, label=labels[k])
for nm, r in BENCH.items():
    ax.plot((1 + r.reindex(span).fillna(0)).cumprod(), lw=1.2, ls="--", label=nm)
ax.set_yscale("log"); ax.set_ylabel("Cumulative wealth (log)"); ax.set_xlabel("Date")
ax.set_title("Cross-Asset Deep Momentum vs Benchmarks (walk-forward OOS)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "equity_curves_05b.png", dpi=150); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for k, r in strat_ret.items():
    cum = (1 + r).cumprod(); dd = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, alpha=0.25)
    ax.plot(dd.index, dd.values, lw=1, label=labels[k])
ax.set_ylabel("Drawdown"); ax.set_xlabel("Date"); ax.set_title("Underwater drawdown")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "drawdown_05b.png", dpi=150); plt.show()


## §9 — Write comparison rows

In [ ]:
def stats_row(name, r, arch):
    return dict(strategy=name, arch=arch, sharpe=round(ann_sharpe(r), 3),
                ann_ret=round(ann_return(r), 4), ann_vol=round(ann_vol(r), 4),
                max_dd=round(max_drawdown(r), 4),
                start=str(r.index[0].date()), end=str(r.index[-1].date()))

comp_rows = [stats_row(labels[k], r, "DeepMomentum") for k, r in strat_ret.items()]
comp_rows += [stats_row(nm, r.reindex(span).dropna(), "Benchmark") for nm, r in BENCH.items()]
comp_df = pd.DataFrame(comp_rows).sort_values("sharpe", ascending=False).reset_index(drop=True)
comp_df.to_csv(OUT_DIR / "comparison.csv", index=False)

# Persist OOS return series for the master table
ret_panel = pd.DataFrame({labels[k]: r for k, r in strat_ret.items()})
ret_panel.to_parquet(OUT_DIR / "deep_momentum_returns.parquet")
print(comp_df.to_string(index=False))
print(f"\nSaved -> {OUT_DIR/'comparison.csv'} and deep_momentum_returns.parquet")


## §10 — Findings (fill after the full run)

> **Headline (template):** Cross-asset deep momentum Sharpe = **___** vs pooled **___** vs
> Risk Parity **___** / TSMOM-12m **___** over the walk-forward OOS span. Cross-asset
> {beats / ties / misses} pooled, {confirming / contradicting} the cross-sectional lift.

**Read-out checklist**
1. **Did the nb-05 null break?** Compare DeepMom(CrossAsset) to Risk Parity *in this universe*. If it
   now wins where the 2-asset version tied, the null was the cage, not the method.
2. **Cross-asset vs pooled.** A cross > pooled gap signals that inter-asset structure pays.
3. **Leaderboard reality check.** Even a win here likely sits *below* the 2.42 / 2.58 covariance-driven
   bars — momentum-only. State that plainly; don't compare across mismatched windows without the caveat.
4. **Robustness before any claim:** rerun with `n_seeds` and `refit_stride` unchanged; the recipe was
   pre-registered in §0. If the result only appears under a tweaked recipe, treat it as fragile.

**Honest scope caveats**
- Sharpe scale here is *not* the repo's 2.x OOS scale — read each number against its stated span and benchmark.
- The delta-of-straddle proxy ($2\Phi(t)-1$) approximates the exact option-delta signal derived from full options data.
- Walk-forward first OOS month is ~10y into the sample; the path is shorter than the full 2003–2026.